# Accuracy and latency

[05_02](05_02_run_vlm.ipynb) ran the model as the paper does and recorded answers and frame times.
First set its accuracy against the paper's own figure and against experiment 04.
Then compare the selectors on accuracy and read time, as [04_02](../04/04_02_accuracy_latency.ipynb) did.

A condition is a method and a requested fraction of the projection weights. `Dense` uses every row.

In [ ]:
from scipy.stats import bootstrap

import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt

## 1. Data

`answers.csv` has one row per question and condition; `correct` says whether the answer was right.
`frame_times.csv` has one row per recorded frame and condition, with times in ms.

In [ ]:
answers     = pd.read_csv("answers.csv")
frame_times = pd.read_csv("frame_times.csv")

## 2. Intervals

Every interval below is a 95% BCa bootstrap with 10,000 resamples, as in the paper.

In [ ]:
def interval(values, statistic):
    """Return the 95% BCa bootstrap interval of a statistic: (low, high)."""

    result = bootstrap(
        (values,),
        statistic,
        confidence_level = 0.95,
        n_resamples      = 10_000,
        method           = "BCa",
        rng              = 0,
    )

    return result.confidence_interval.low, result.confidence_interval.high

## 3. Accuracy

Accuracy is the percentage of right answers of a condition.

In [ ]:
records = []

for (method, fraction), rows in answers.groupby(["method", "fraction"], sort=False):
    scores    = 100.0 * rows.correct.to_numpy()
    low, high = interval(scores, np.mean)

    records.append({
        "method"        : method,
        "fraction"      : fraction,
        "accuracy"      : scores.mean(),
        "accuracy_low"  : low,
        "accuracy_high" : high,
    })

accuracy = pd.DataFrame(records)
accuracy = accuracy.set_index(["method", "fraction"])

## 4. Against the paper

Figure 6 of the paper has a panel for this model and benchmark, measured on a Jetson Orin Nano.
It plots accuracy at sparsity 0% to 70%, and its sparsity is $1 - f$ for a requested fraction $f$.
Its `Baseline` is `Top-R` here and its `Ours` is `Paper greedy`.

The values below are the coordinates of the points of that panel in the PDF.
The paper evaluates on the 385 videos outside its calibration set; 05_02 on 100 of them.

In [ ]:
# Accuracy in the paper (%), by requested fraction
paper = pd.DataFrame(
    {
        "Top-R"        : [53.29, 53.15, 52.33, 52.33, 51.99, 49.73, 48.29, 47.67],
        "Paper greedy" : [53.29, 52.67, 52.33, 51.03, 50.07, 49.79, 47.33, 45.89],
    },
    index = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3],
)

Experiment 04 answered the same questions.
There every projection read the same fraction of its rows, all in checkpoint order.

In [ ]:
answers_04 = pd.read_csv("../04/answers.csv")

# (method, fraction) -> accuracy in experiment 04 (%)
accuracy_04 = 100.0 * answers_04.groupby(["method", "fraction"]).correct.mean()

The accuracy of the paper's two methods at the fractions that 05_02 ran:

In [ ]:
records = []

for method in paper.columns:
    for fraction, row in accuracy.loc[method].iterrows():
        records.append({
            "method"   : method,
            "fraction" : fraction,

            "paper" : paper.loc[fraction, method],
            "04"    : accuracy_04[(method, fraction)],

            "here" : row.accuracy,
            "low"  : row.accuracy_low,
            "high" : row.accuracy_high,
        })

comparison = pd.DataFrame(records)
comparison = comparison.set_index(["method", "fraction"])

comparison.round(1)

One panel per method. Each line runs from the smallest fraction up to `Dense` at 100%.
Vertical bars are the accuracy intervals of 05_02.

In [ ]:
colors = {
    "Top-R"        : "#2a78d6",
    "Paper greedy" : "#eb6834",
    "Tiles"        : "#1baf7a",
}

markers = {
    "Top-R"        : "o",
    "Paper greedy" : "s",
    "Tiles"        : "^",
}

In [ ]:
def draw_paper(ax, method):
    values = paper[method]

    ax.plot(
        100 * values.index,
        values,
        label = "Paper",

        color           = "gray",
        linestyle       = "--",
        marker          = "o",
        markerfacecolor = "white",
    )

In [ ]:
def draw_04(ax, method):

    # The fractions of the method, then Dense at 100%
    values = pd.concat([accuracy_04.loc[method], accuracy_04.loc["Dense"]])

    ax.plot(
        100 * values.index,
        values,
        label = "Experiment 04",

        color     = colors[method],
        linestyle = ":",
        marker    = "x",
    )

In [ ]:
def draw_here(ax, method):

    # The fractions of the method, then Dense at 100%
    rows  = pd.concat([accuracy.loc[method], accuracy.loc["Dense"]])
    below = rows.accuracy - rows.accuracy_low
    above = rows.accuracy_high - rows.accuracy

    ax.errorbar(
        100 * rows.index,
        rows.accuracy,
        yerr  = [below, above],
        label = "Experiment 05",

        color      = colors[method],
        marker     = markers[method],
        markersize = 8,
        linewidth  = 2,
        capsize    = 3,
    )

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize = (11, 4.5),
    sharey  = True,
    layout  = "constrained",
)

for ax, method in zip(axes, paper.columns):
    draw_paper(ax, method)
    draw_04(ax, method)
    draw_here(ax, method)

    ax.set(
        title  = method,
        xlabel = "Projection weights requested (%)",
    )

    ax.grid(color="gainsboro", linewidth=0.6)
    ax.legend(loc="lower right")

axes[0].set_ylabel("Accuracy (%)")

questions = answers.question_id.nunique()
fig.suptitle(f"LLaVA-OneVision-Qwen2-0.5B · TempCompass · {questions} questions")

plt.show()

## 5. Time per frame

For one frame, `selection_ms` is the time the selectors of all projections take
and `io_ms` the time to read the selected rows from flash.
Their sum, `load_ms`, is how long the frame waits for its weights.
Take the median of each over the recorded frames.

In [ ]:
frame_times["load_ms"] = frame_times.selection_ms + frame_times.io_ms

In [ ]:
records = []

for (method, fraction), rows in frame_times.groupby(["method", "fraction"], sort=False):
    low, high = interval(rows.load_ms.to_numpy(), np.median)

    records.append({
        "method"       : method,
        "fraction"     : fraction,
        "selection_ms" : rows.selection_ms.median(),
        "io_ms"        : rows.io_ms.median(),
        "load_ms"      : rows.load_ms.median(),
        "load_low"     : low,
        "load_high"    : high,
    })

latency = pd.DataFrame(records)
latency = latency.set_index(["method", "fraction"])

## 6. Summary

In [ ]:
summary = accuracy.join(latency)
summary.round(1)

## 7. Plot

Each line connects the three fractions of one method.
Vertical bars are the accuracy intervals; the label at one end of a bar gives the fraction.
The dotted lines cross at `Dense`: points to their right take longer than reading every row.

Left: read time only. Right: selection and read time. The time axes are logarithmic.

In [ ]:
# The fraction label of a point sits at one end of its accuracy interval:
# (end, vertical alignment, points up from that end).
# Paper and Tiles lie close together, so their labels take opposite ends.
anchors = {
    "Top-R"        : ("accuracy_high", "bottom",  3),
    "Paper greedy" : ("accuracy_high", "bottom",  3),
    "Tiles"        : ("accuracy_low",  "top",    -3),
}

In [ ]:
def draw_dense(ax, column):
    dense = summary.loc[("Dense", 1.0)]

    ax.axvline(dense[column],  color="gray", linestyle=":", linewidth=1)
    ax.axhline(dense.accuracy, color="gray", linestyle=":", linewidth=1)

    ax.plot(
        dense[column],
        dense.accuracy,
        label = "Dense",

        color     = "black",
        linestyle = "none",
        marker    = "D",
    )

In [ ]:
def draw_method(ax, column, method):
    rows  = summary.loc[method]
    below = rows.accuracy - rows.accuracy_low
    above = rows.accuracy_high - rows.accuracy

    ax.errorbar(
        rows[column],
        rows.accuracy,
        yerr  = [below, above],
        label = method,

        color      = colors[method],
        marker     = markers[method],
        markersize = 8,
        linewidth  = 2,
        capsize    = 3,
    )

    # Fraction labels
    end, va, shift = anchors[method]

    for fraction, row in rows.iterrows():
        ax.annotate(
            f"{fraction:.0%}",
            (row[column], row[end]),
            xytext     = (0, shift),
            textcoords = "offset points",
            ha         = "center",
            va         = va,
            color      = "dimgray",
            fontsize   = 8,
        )

In [ ]:
TICKS = [100, 200, 400, 800]  # Ticks of the time axes (ms)

def draw_panel(ax, column, label):
    draw_dense(ax, column)

    for method in colors:
        draw_method(ax, column, method)

    ax.set(
        xscale = "log",
        xlabel = label,
    )

    # Plain numbers at the ticks instead of powers of ten
    ax.set_xticks(TICKS, labels=TICKS)
    ax.minorticks_off()

    # Room for the fraction labels
    ax.margins(y=0.08)

    ax.grid(color="gainsboro", linewidth=0.6)

In [ ]:
fig, (left, right) = plt.subplots(
    1, 2,
    figsize = (11, 4.5),
    sharey  = True,
    layout  = "constrained",
)

draw_panel(left,  "io_ms",   "Read time per frame (ms)")
draw_panel(right, "load_ms", "Selection + read time per frame (ms)")

left.set_ylabel("Accuracy (%)")

handles, labels = left.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=4)

fig.suptitle(f"LLaVA-OneVision-Qwen2-0.5B · TempCompass · {questions} questions")

plt.show()